# 10 — Step 8: onboarding a new user (the cold-start problem)

**Problem.** CF needs a user's ratings, and a new VibeRec user has none, so the best recommender (ALS + vibe) falls back to popularity. PROJECT.md §6: a short **like / dislike / haven't-seen questionnaire** gives the system its first ratings. Which movies should it ask about, and how many questions does it take to get good recommendations? **Pass/fail check (roadmap):** how quickly new-user recommendations stabilise.

**Simulating a new user.** Take a testing-zone user (2018 cutoff) and hide their whole history. Ask questionnaire movies one at a time; their real pre-2018 ratings answer:
- **like** = rated above their own average;
- **dislike** = at or below it;
- **haven't seen** = never rated it.

After **5, 10, 20 and 30** questions, the user is folded into ALS from the liked answers (the exact ALS user step, `cf.fold_in_user`, checked against the `implicit` library to 4 decimals). They then get the ALS + vibe blend (w_ALS = 0.2 from Step 7), scored against their real future favourites. With no likes yet, they get popularity. The system does not know the user's hidden history, so it may recommend movies they had already seen; that counts as a miss, as it would in real use.

**Fairness.** ALS is retrained **without the 5,000 test users**, so they are genuinely new to it.

**Question strategies.** All draw from the same candidates: the 2,000 most-rated tier-3 movies at the cutoff (a computing setting; asking about obscure films mostly gets "haven't seen").

| Strategy | Picks | Role |
|---|---|---|
| random | random candidates | floor |
| popular | most-rated first | the obvious approach |
| informative | one fixed list chosen by optimal experiment design: the movies whose answers best pin down a user's ALS taste factors, each weighted by P(seen) × like-uncertainty | the data-driven fixed list; no hand-set weights |
| adaptive | per user, the next question that adds the most information given the answers so far, with P(seen) updated from what this user has already seen (co-watching patterns) | the smart version |

**Two measurements** (revised after the first run, see section 3):
1. **Learning the user's taste (main):** do the recommendations rank the user's **other liked movies**, i.e. their hidden pre-2018 likes minus the movies asked about? This is the standard cold-start test and the roadmap's check.
2. **Future hits:** the testing-zone measure (movies rated after the 2018 cutoff above the user's own average).

**Reference: no questions at all** = the most popular movies, with the system knowing nothing about the user.

## 1. Load, and ALS without the test users

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))
from src.viberec import cf, evaluation as ev, onboarding as ob, recommenders as rc

PROCESSED_DIR, ARTIFACTS_DIR, MODELS_DIR = Path("..") / "data" / "processed", Path("..") / "artifacts", Path("..") / "models"
SCOREBOARD = ARTIFACTS_DIR / "scoreboard.csv"
VERSION, SEED, N_CANDIDATES, CHECKPOINTS = "v1", 42, 2000, (5, 10, 20, 30)


def write_metadata(path, **info):
    meta = {"created_at": datetime.now(timezone.utc).isoformat(), "produced_by": "notebooks/10_onboarding.ipynb", **info}
    path.with_name(path.stem + ".metadata.json").write_text(json.dumps(meta, indent=2, default=str))


pre2018 = pd.read_parquet(PROCESSED_DIR / f"ratings_before_2018_{VERSION}.parquet")
test_train = pd.read_parquet(PROCESSED_DIR / f"benchmark_train_{VERSION}.parquet")
test_hits = pd.read_parquet(PROCESSED_DIR / f"benchmark_hits_{VERSION}.parquet").groupby("userId")["movieId"].apply(set).to_dict()
test_mean = test_train.groupby("userId")["rating"].mean()
at_cutoff = pd.read_parquet(PROCESSED_DIR / f"benchmark_movies_at_cutoff_{VERSION}.parquet").set_index("movieId")
pool = at_cutoff.index[at_cutoff["tier"] == 3].to_numpy()
vibes = pd.read_parquet(PROCESSED_DIR / "vibe_scores_v2.parquet").set_index("movieId").drop(columns="source").astype(float)
tuning = pd.read_csv(ARTIFACTS_DIR / f"cf_tuning_{VERSION}.csv")
best_als = json.loads(tuning[tuning["method"] == "als"].sort_values(["recall@10", "recall@50"], ascending=False)["params"].iloc[0])
blend_tuning = pd.read_csv(ARTIFACTS_DIR / f"cf_vibe_blend_tuning_{VERSION}.csv")
W_ALS = float(blend_tuning.sort_values(["recall@10", "recall@50"], ascending=False)["w_als"].iloc[0])
print("ALS settings:", best_als, "| blend w_als:", W_ALS)

others = pre2018[~pre2018["userId"].isin(test_hits.keys())]               # everyone except the test users
matrix, u_index, m_index = cf.build_matrix(others, best_als["table"])
ALS_PATH = MODELS_DIR / f"als_item_factors_no_test_users_{VERSION}.npy"
if ALS_PATH.exists():
    print(f"{ALS_PATH} already exists, loading.")
    item_factors = np.load(ALS_PATH)
    assert len(item_factors) == len(m_index)
else:
    from implicit.als import AlternatingLeastSquares
    start = time.time()
    model = AlternatingLeastSquares(factors=best_als["factors"], regularization=best_als["reg"], alpha=best_als["alpha"],
                                    iterations=15, random_state=SEED)
    model.fit(matrix, show_progress=False)
    item_factors = np.asarray(model.item_factors)
    np.save(ALS_PATH, item_factors)
    write_metadata(ALS_PATH, params=best_als, trained_on="ratings before 2018-01-04, test users excluded",
                   users=len(u_index), movies=len(m_index), runtime_seconds=round(time.time() - start, 1),
                   note="row order = sorted movieIds of those ratings (rebuilt by cf.build_matrix)")
yty = item_factors.T @ item_factors
print(f"ALS on {len(u_index):,} users x {len(m_index):,} movies (test users excluded)")

## 2. Question candidates and what the strategies need

Computed from users other than the test users, before the cutoff. For each of the 2,000 most-rated tier-3 movies:
- **P(seen):** share of users who rated it;
- **like share:** among those who rated it, the share who rated it above their own average. Like-uncertainty p(1 − p) is highest for divisive movies;
- **co-watching:** P(seen i | seen j), used by the adaptive strategy.

In [ ]:
cand = at_cutoff.loc[pool].nlargest(N_CANDIDATES, "rating_count").index.intersection(m_index.index)
watched, _, cmi = cf.build_matrix(others, "watched", user_ids=u_index.index, movie_ids=cand)
liked, _, _ = cf.build_matrix(others, "liked", user_ids=u_index.index, movie_ids=cand)
n_seen = np.asarray(watched.sum(axis=0)).ravel()
seen_share = n_seen / matrix.shape[0]
like_share = np.asarray(liked.sum(axis=0)).ravel() / np.maximum(n_seen, 1)
co = (watched.T @ watched).toarray()
cowatch = co / np.maximum(np.diag(co), 1)[None, :]                          # column j: P(seen i | seen j)
q = ob.Questionnaire(cand.to_numpy(), item_factors[m_index.loc[cand].to_numpy()], seen_share, like_share, cowatch,
                     reg=best_als["reg"])
titles = pd.read_parquet(PROCESSED_DIR / "movie_tiers_v2.parquet").set_index("movieId")["title"]
fixed30 = q.fixed_order("informative", 30)
print(f"{len(cand):,} candidates; P(seen) range {seen_share.min():.3f}-{seen_share.max():.3f}")
print("\nthe informative fixed questionnaire (in order), with P(seen) and like share:")
for rank, i in enumerate(fixed30, 1):
    print(f"{rank:2d}. {titles.get(cand[i], cand[i])[:45]:45s} seen {seen_share[i]:.2f}  liked {like_share[i]:.2f}")

## 3. Simulate the questionnaire for the 5,000 test users

Each strategy asks up to 30 questions per user. The adaptive strategy's first n answers are exactly what it would have asked with only n questions, so every checkpoint comes from the same run.

**Why two measurements (lesson from the first run).**
- The first run (`onboarding_results_v1`) scored only future hits and compared them with the testing-zone popularity row (0.0229). That row knows each user's history and skips movies they have seen; a new user's history is unknown.
- These test users have rated a median of 283 movies, so a newcomer's list is full of classics they have already seen, which can never be future hits.
- The fair "no questions" baseline is popularity *without* history: 0.0122 on future hits, not 0.0229.
- The main measurement is therefore **learning the user's taste**: recall of their hidden pre-2018 liked movies (median 155 per user), excluding the movies asked about.

Results: `artifacts/onboarding_results_v2.parquet`.

In [ ]:
pool_ids = np.asarray([m for m in pool if m in m_index.index])
pool_cols = m_index.loc[pool_ids].to_numpy()
z = rc.standardise(vibes)
vibe_unit = z.div(np.linalg.norm(z.to_numpy(), axis=1), axis=0).reindex(pool_ids).to_numpy()   # NaN where no vibes
popular_order = pool_ids[np.argsort(-at_cutoff.loc[pool_ids, "rating_count"].to_numpy())]


def standardised(s):
    finite = np.isfinite(s)
    if not finite.any():
        return np.zeros_like(s)
    return np.where(finite, (s - s[finite].mean()) / (s[finite].std() or 1.0), 0.0)


def recommend_new_user(answers, k=50, use_dislikes=False, exclude=()):
    """ALS + vibe blend from questionnaire answers only; popularity if nothing was liked yet.
    use_dislikes: the vibe taste becomes mean(liked vibes) - mean(disliked vibes) (section 6 test).
    exclude: extra movies never to recommend (section 6 grades every plan outside the whole question list)."""
    asked = {m for m, _ in answers} | set(exclude)
    likes = [m for m, a in answers if a == "like" and m in m_index.index]
    if not likes:
        return [m for m in popular_order if m not in asked][:k]
    als = item_factors[pool_cols] @ cf.fold_in_user(item_factors, yty, m_index.loc[likes].to_numpy(), best_als["alpha"], best_als["reg"])
    like_vibes = [m for m in likes if m in z.index]
    taste = z.loc[like_vibes].mean().to_numpy() if like_vibes else None
    dislike_vibes = [m for m, a in answers if a == "dislike" and m in z.index]
    if use_dislikes and taste is not None and dislike_vibes:
        taste = taste - z.loc[dislike_vibes].mean().to_numpy()
    vib = vibe_unit @ (taste / (np.linalg.norm(taste) or 1)) if taste is not None else np.full(len(pool_ids), np.nan)
    score = W_ALS * standardised(als) + (1 - W_ALS) * standardised(vib)
    score[np.isin(pool_ids, list(asked))] = -np.inf
    top = np.argpartition(-score, k)[:k]
    return list(pool_ids[top[np.argsort(-score[top])]])


RES_PATH = ARTIFACTS_DIR / "onboarding_results_v2.parquet"               # v2: adds the taste-learning measurement
if RES_PATH.exists():
    print(f"{RES_PATH} already exists, loading.")
    onb = pd.read_parquet(RES_PATH)
else:
    start, rows = time.time(), []
    by_user = dict(tuple(test_train.groupby("userId")))
    hidden_likes = {u: set(r.loc[r["rating"] > test_mean[u], "movieId"]) for u, r in by_user.items()}

    def add_row(strategy, n, user, ans, hits_u):
        recs = recommend_new_user(ans)
        asked = {m for m, _ in ans}
        profile = hidden_likes[user] - asked                                  # the likes the system was not told
        rows.append({"strategy": strategy, "n_questions": n, "userId": user,
                     "answered_seen": sum(a is not None for _, a in ans), "likes": sum(a == "like" for _, a in ans),
                     "taste_recall@10": ev.recall_at_k(recs, profile, 10), "taste_recall@50": ev.recall_at_k(recs, profile, 50),
                     "recall@10": ev.recall_at_k(recs, hits_u, 10), "recall@50": ev.recall_at_k(recs, hits_u, 50),
                     "ndcg@10": ev.ndcg_at_k(recs, hits_u, 10)})

    for user, hits_u in test_hits.items():                                    # no questions: popularity, history unknown
        add_row("no questions", 0, user, [], hits_u)
    for s_i, strategy in enumerate(("random", "popular", "informative", "adaptive")):
        for user, hits_u in test_hits.items():
            answer = ob.answer_from_history(by_user[user], test_mean[user])
            run = q.run(strategy, answer, max(CHECKPOINTS), seed=SEED + int(user))
            for n in CHECKPOINTS:
                add_row(strategy, n, user, run[:n], hits_u)
        print(f"{strategy} done, {time.time() - start:.0f}s")
    onb = pd.DataFrame(rows)
    onb.to_parquet(RES_PATH, index=False)
    write_metadata(RES_PATH, users=len(test_hits), checkpoints=CHECKPOINTS, candidates=N_CANDIDATES, w_als=W_ALS,
                   als=best_als, runtime_seconds=round(time.time() - start, 1))

summary = onb.groupby(["strategy", "n_questions"])[["answered_seen", "likes", "taste_recall@10", "taste_recall@50",
                                                     "recall@10", "recall@50", "ndcg@10"]].mean()
print(summary.round(4).to_string())

## 4. Which strategy learns the user's taste fastest? (paired, user by user)

- Every strategy at every checkpoint is compared with **no questions** (popularity, history unknown): does asking help at all?
- At the same number of questions, each strategy is compared with **popular** questions: does a smarter choice help?

Both are run on the main measurement (taste recall@10) and on future hits. Rows go to the scoreboard.

In [ ]:
per = onb.set_index("userId")
none = per[per["strategy"] == "no questions"]
for metric in ("taste_recall@10", "recall@10"):
    print(f"--- {metric} ---  (no questions: {none[metric].mean():.4f})")
    for n in CHECKPOINTS:
        parts = []
        base = per[(per["strategy"] == "popular") & (per["n_questions"] == n)]
        for strategy in ("random", "popular", "informative", "adaptive"):
            cur = per[(per["strategy"] == strategy) & (per["n_questions"] == n)]
            d, lo, hi, rel = ev.paired_gain(cur, none, metric, seed=SEED)
            txt = f"{strategy} {cur[metric].mean():.4f} ({rel:+.0%} vs none [{lo:+.4f}, {hi:+.4f}]"
            if strategy != "popular":
                d2, lo2, hi2, rel2 = ev.paired_gain(cur, base, metric, seed=SEED)
                txt += f"; {rel2:+.0%} vs popular questions [{lo2:+.4f}, {hi2:+.4f}]"
            parts.append(txt + ")")
        print(f"{n:2d} answers: " + "\n            ".join(parts))

for (strategy, n), r in summary.iterrows():
    res = {"taste_recall@10": r["taste_recall@10"], "taste_recall@50": r["taste_recall@50"], "recall@10": r["recall@10"],
           "recall@50": r["recall@50"], "ndcg@10": r["ndcg@10"], "answered_seen": r["answered_seen"], "users": len(test_hits)}
    ev.append_scoreboard(SCOREBOARD, "8 onboarding", f"new user: {strategy}, {n} asked (history unknown)", res,
                         settings={"cutoff": "2018-01-04", "users": len(test_hits), "seed": SEED, "candidates": N_CANDIDATES,
                                   "strategy": strategy, "n_questions": int(n), "w_als": W_ALS, "protocol": "v2"},
                         note="history hidden; taste_recall = hidden pre-2018 likes; recall = future hits")

**Results (5,000 simulated new users, history hidden)**

**Learning the user's taste** (recall@10 of their hidden pre-2018 likes; no questions = 0.0393):

| Questions answered | random | popular | **informative (fixed)** | adaptive |
|---|---|---|---|---|
| 5 | 0.0379 | 0.0374 | 0.0390 | 0.0387 |
| 10 | 0.0370 | **0.0433** (+12%) | 0.0415 (+6%) | 0.0401 |
| 20 | 0.0371 | **0.0474** (+24%) | 0.0454 (+19%) | 0.0454 (+18%) |
| 30 | 0.0376 | 0.0449 (+19%) | **0.0529 (+38%)** | 0.0505 (+32%) |

Percentages are against no questions; all those shown are significant (paired intervals exclude 0).

- **Asking works.** With 30 well-chosen questions the system ranks the user's other favourites 38% better than knowing nothing.
- **Random questions never help:** a user has seen only about 4 of 30 random movies, so there is almost nothing to learn from.
- **The informative fixed list wins at 30 questions:**
  - +15% over popular questions [+0.0059, +0.0079];
  - +5% over adaptive.
  - At 10–20 questions popular questions are slightly ahead, because more of them have been seen. The informative list trades a slower start for more information per answer.
- **Adaptive does not beat the fixed informative list** (0.0505 vs 0.0529 at 30), so by the simpler-wins rule the questionnaire is **static**. That answers the open question in PROJECT.md §14.
- **Not yet stable at 30:** the informative curve is still rising (0.039 → 0.042 → 0.045 → 0.053), so more answers would keep helping. Feedback after onboarding (thumbs up/down, §6) can continue that.

**Future hits** (no questions = 0.0122, popularity without history): no strategy is clearly better than no questions at any length (all intervals include 0, except adaptive at 30, which is slightly worse than popular questions).
- Why: these test users had rated a median of 283 movies, so a newcomer's list is full of classics they have already seen, which can never be future hits.
- Real new VibeRec users will have seen far fewer movies, so this penalty is specific to the simulation. Learning their taste is the part onboarding controls, and that measurably improves.

**The questionnaire (data-chosen, for Adil to review).** The first 30 informative questions are widely seen 1970s–2000s films spread across vibes: Forrest Gump, Toy Story, Braveheart, Jurassic Park, Pulp Fiction, Schindler's List, Star Wars, The Silence of the Lambs, 12 Monkeys, Apollo 13, The Matrix, Independence Day, Terminator 2, Aladdin, The Shawshank Redemption, Batman, Dances with Wolves, Back to the Future, Fargo, The Godfather, American Beauty, True Lies, Titanic, The Fellowship of the Ring, The Fugitive, Raiders of the Lost Ark, Saving Private Ryan, The Lion King, The Usual Suspects, Mission: Impossible. It includes divisive ones (Independence Day, Batman, Titanic and Mission: Impossible are liked by only 40–46% of those who saw them).

**Limitations**
- Simulated users are heavy MovieLens raters, and the list leans on 1990s films because MovieLens does.
- Taste recall rewards learning a profile the user already has; it is a proxy for future satisfaction.
- The ALS + vibe blend weight was tuned for known users, not new ones.

## 5. A questionnaire for today's audience (Adil's feedback)

**Feedback (Adil, 2026-10-09).** The list is too 1990s. A 2026 user is more likely to know post-2000 films, unless an older one is extremely famous.

**Evidence.** Users who joined MovieLens after mid-2016 (10,903) put 64% of their ratings on 2000s–2010s films; earlier users put 25% there. P(seen) estimated from all users therefore reflects an older audience.

**Two variants, tested against the current list:**
- **A, recent audience:** P(seen) from users who joined after mid-2016 only. Like-uncertainty still comes from all users, for stable estimates. An old film ranks high only if new audiences still watch it, so "unless it's too famous" is decided by data, not by a hand-set rule.
- **B, 2000+ only:** a hard cut to films released in 2000 or later.

Same informative design and same simulation (taste recall, 5,000 users, history hidden).

In [ ]:
VAL_CUTOFF = int(pd.read_parquet(ARTIFACTS_DIR / f"q43_early_counts_{VERSION}.parquet").query("quantile == 0.8")["cutoff_ts"].iloc[0])
first_rating = others.groupby("userId")["timestamp"].min()
recent_users = first_rating.index[first_rating >= VAL_CUTOFF]
recent_watched, _, _ = cf.build_matrix(others[others["userId"].isin(recent_users)], "watched",
                                       user_ids=recent_users.sort_values(), movie_ids=cand)
seen_recent = np.asarray(recent_watched.sum(axis=0)).ravel() / len(recent_users)
years = pd.read_parquet(PROCESSED_DIR / "movie_tiers_v2.parquet").set_index("movieId")["year"]
cand_years = years.reindex(cand).to_numpy()
factors_c = item_factors[m_index.loc[cand].to_numpy()]

variants = {"current (all users)": ob.Questionnaire(cand.to_numpy(), factors_c, seen_share, like_share, cowatch, best_als["reg"]),
            "A: recent audience": ob.Questionnaire(cand.to_numpy(), factors_c, seen_recent, like_share, cowatch, best_als["reg"])}
modern = cand_years >= 2000
variants["B: 2000+ only"] = ob.Questionnaire(cand.to_numpy()[modern], factors_c[modern], seen_share[modern], like_share[modern],
                                             cowatch[np.ix_(modern, modern)], best_als["reg"])

lists = {name: [qq.ids[i] for i in qq.fixed_order("informative", 30)] for name, qq in variants.items()}
for name, ids in lists.items():
    y = years.reindex(ids)
    print(f"{name}: median release year {int(y.median())}, {int((y >= 2000).sum())} of 30 from 2000+")
    print("   " + " · ".join(titles.get(m, str(m))[:32] for m in ids))

V_PATH = ARTIFACTS_DIR / "onboarding_variants_v1.parquet"
if V_PATH.exists():
    print(f"\n{V_PATH} already exists, loading.")
    var = pd.read_parquet(V_PATH)
else:
    by_user = dict(tuple(test_train.groupby("userId")))
    hidden_likes = {u: set(r.loc[r["rating"] > test_mean[u], "movieId"]) for u, r in by_user.items()}
    rows = []
    for name, ids in lists.items():
        for user in test_hits:
            answer = ob.answer_from_history(by_user[user], test_mean[user])
            run = [(m, answer(m)) for m in ids]
            for n in (10, 20, 30):
                ans = run[:n]
                profile = hidden_likes[user] - {m for m, _ in ans}
                recs = recommend_new_user(ans)
                rows.append({"variant": name, "n_questions": n, "userId": user, "answered_seen": sum(a is not None for _, a in ans),
                             "taste_recall@10": ev.recall_at_k(recs, profile, 10), "recall@10": ev.recall_at_k(recs, test_hits[user], 10)})
    var = pd.DataFrame(rows)
    var.to_parquet(V_PATH, index=False)
    write_metadata(V_PATH, variants=list(lists), lists={k: [int(m) for m in v] for k, v in lists.items()},
                   recent_users=len(recent_users), note="informative fixed questionnaire: all-user vs recent-user P(seen) vs 2000+ only")

print()
print(var.groupby(["variant", "n_questions"])[["answered_seen", "taste_recall@10", "recall@10"]].mean().round(4).to_string())
cur = var[(var["variant"] == "current (all users)")].set_index(["n_questions", "userId"])
for name in ("A: recent audience", "B: 2000+ only"):
    other = var[var["variant"] == name].set_index(["n_questions", "userId"])
    for n in (10, 20, 30):
        d, lo, hi, rel = ev.paired_gain(other.loc[n], cur.loc[n], "taste_recall@10", seed=SEED)
        print(f"{name} vs current, {n} answers: taste recall@10 {rel:+.0%} [{lo:+.4f}, {hi:+.4f}]")

### Adil's rule: about 25 films from 2000+ and 5 famous 1990s films

**Requirement (Adil, 2026-10-09).** A new user should mostly be asked about post-2000 films, with a few famous 1990s ones. This is a product rule: at most **5 from the 1990s**, the other **25 from 2000+**; no pre-1990 films.

**Within that rule, the data still chooses.** Same informative design, P(seen) from the recent audience (variant A, which tested best). Picks are made jointly, so the order still puts the most useful question first.

In [ ]:
decade = np.where(cand_years >= 2000, "2000+", np.where(cand_years >= 1990, "1990s", "before 1990"))
QUOTA = {"2000+": 25, "1990s": 5}
q_rule = ob.Questionnaire(cand.to_numpy(), factors_c, seen_recent, like_share, cowatch, best_als["reg"],
                          groups=decade, quota=QUOTA)
rule_list = [q_rule.ids[i] for i in q_rule.fixed_order("informative", 30)]
print("C: 25 from 2000+ / 5 from the 1990s (recent audience):")
for rank, m in enumerate(rule_list, 1):
    i = int(np.where(cand.to_numpy() == m)[0][0])
    print(f"{rank:2d}. {titles.get(m, str(m))[:45]:45s} seen (recent users) {seen_recent[i]:.2f}  liked {like_share[i]:.2f}")

C_PATH = ARTIFACTS_DIR / "onboarding_variant_quota_v1.parquet"
if C_PATH.exists():
    print(f"\n{C_PATH} already exists, loading.")
    var_c = pd.read_parquet(C_PATH)
else:
    by_user = dict(tuple(test_train.groupby("userId")))
    hidden_likes = {u: set(r.loc[r["rating"] > test_mean[u], "movieId"]) for u, r in by_user.items()}
    rows = []
    for user in test_hits:
        answer = ob.answer_from_history(by_user[user], test_mean[user])
        run = [(m, answer(m)) for m in rule_list]
        for n in (10, 20, 30):
            ans = run[:n]
            recs = recommend_new_user(ans)
            rows.append({"variant": "C: 25 x 2000+ / 5 x 1990s", "n_questions": n, "userId": user,
                         "answered_seen": sum(a is not None for _, a in ans),
                         "taste_recall@10": ev.recall_at_k(recs, hidden_likes[user] - {m for m, _ in ans}, 10),
                         "recall@10": ev.recall_at_k(recs, test_hits[user], 10)})
    var_c = pd.DataFrame(rows)
    var_c.to_parquet(C_PATH, index=False)
    write_metadata(C_PATH, quota=QUOTA, questionnaire=[int(m) for m in rule_list],
                   note="Adil's rule: at most 5 films from the 1990s, 25 from 2000+, none before 1990; informative design, recent-audience P(seen)")

allv = pd.concat([var, var_c])
print()
print(allv.groupby(["variant", "n_questions"])[["answered_seen", "taste_recall@10", "recall@10"]].mean().round(4).to_string())
c_idx = var_c.set_index(["n_questions", "userId"])
for ref in ("current (all users)", "A: recent audience"):
    r_idx = var[var["variant"] == ref].set_index(["n_questions", "userId"])
    for n in (10, 20, 30):
        d, lo, hi, rel = ev.paired_gain(c_idx.loc[n], r_idx.loc[n], "taste_recall@10", seed=SEED)
        print(f"C vs {ref}, {n} answers: taste recall@10 {rel:+.0%} [{lo:+.4f}, {hi:+.4f}]")

## 6. Keep asking until the user has liked enough movies (Adil's idea)

**Idea (Adil, 2026-10-09).** Ask from a longer list (60–100 movies) until the user has liked 20–30 of them; answers are like / dislike / no clue. Likes are what fold a user into ALS, so stopping on likes targets the right thing, and taste learning was still improving at 30 questions.

**The extended list**
- 100 movies with the same rule share (83 from 2000+, 17 from the 1990s, none earlier) and the same informative design.
- It **starts with the official 30, unchanged**; questions 31–100 are chosen knowing those were already asked.

**Compared (taste recall@10 and future-hit recall@10, paired by user):**
- **stop at 20 likes** and **stop at 30 likes** (or when the list runs out);
- **fixed 30**, the official list;
- **fixed, same length:** a fixed number of questions equal to the median that "stop at 20 likes" needed. This separates "stopping on likes" from simply "asking more";
- **stop at 20 likes + dislikes used:** the vibe taste is moved away from disliked movies' vibes (mean of liked − mean of disliked; the simplest form, a first test rather than a tuned rule).

**Also measured:** how many questions each user needed (median and the slowest 10%), and how many never reach the target within 100. The simulated users are heavy raters (median 283 movies), so real users will need more questions than shown here.

**Grading note (v3).** Every plan recommends **and** is graded outside the whole 100-movie list, so all plans compete on exactly the same movies and the same target: the user's pre-2018 likes that are not on the list. Two earlier gradings were biased and are kept only for the record:
- **v1** removed only the asked movies from the target, so longer plans were graded on a harder target;
- **v2** fixed the target but not the recommendations, so shorter plans wasted top-10 slots on list movies that could never count.

In [ ]:
EXT_QUOTA = {"2000+": 83, "1990s": 17}
start_rows = [int(np.where(cand.to_numpy() == m)[0][0]) for m in rule_list]
ext_rows = cf.informative_questions(factors_c, seen_recent * q_rule.like_unc, 100, best_als["reg"],
                                    groups=decade, quota=EXT_QUOTA, start=start_rows)
ext_list = list(cand.to_numpy()[ext_rows])
assert ext_list[:30] == rule_list
print(f"extended list: {len(ext_list)} movies, {sum(years.get(m, 0) >= 2000 for m in ext_list)} from 2000+; questions 31-45:")
print("   " + " · ".join(titles.get(m, str(m))[:30] for m in ext_list[30:45]))

# v3: every plan recommends and is graded OUTSIDE the whole 100-movie list (same candidates, same target for all).
# v1 removed only asked movies from the target (longer plans graded harder); v2 fixed the target but not the
# recommendations (shorter plans wasted top-10 slots on list movies that could never count). Both kept, not used.
S_PATH = ARTIFACTS_DIR / "onboarding_stop_on_likes_v3.parquet"
if S_PATH.exists():
    print(f"\n{S_PATH} already exists, loading.")
    stop = pd.read_parquet(S_PATH)
else:
    by_user = dict(tuple(test_train.groupby("userId")))
    hidden_likes = {u: set(r.loc[r["rating"] > test_mean[u], "movieId"]) for u, r in by_user.items()}
    list_set = set(ext_list)
    runs, need = {}, {20: [], 30: []}
    for user in test_hits:
        answer = ob.answer_from_history(by_user[user], test_mean[user])
        runs[user] = [(m, answer(m)) for m in ext_list]
        likes_cum = np.cumsum([a == "like" for _, a in runs[user]])
        for target in (20, 30):
            hit = np.where(likes_cum >= target)[0]
            need[target].append(int(hit[0]) + 1 if len(hit) else len(ext_list))
    same_len = int(np.median(need[20]))
    plans = {"stop at 20 likes": ({u: n for u, n in zip(test_hits, need[20])}, False),
             "stop at 30 likes": ({u: n for u, n in zip(test_hits, need[30])}, False),
             "fixed 30 (official)": ({u: 30 for u in test_hits}, False),
             f"fixed {same_len} (same median length)": ({u: same_len for u in test_hits}, False),
             "stop at 20 likes + dislikes": ({u: n for u, n in zip(test_hits, need[20])}, True)}
    rows = []
    for plan, (length, use_dis) in plans.items():
        for user in test_hits:
            ans = runs[user][:length[user]]
            recs = recommend_new_user(ans, use_dislikes=use_dis, exclude=list_set)
            rows.append({"plan": plan, "userId": user, "questions": length[user], "likes": sum(a == "like" for _, a in ans),
                         "taste_recall@10": ev.recall_at_k(recs, hidden_likes[user] - list_set, 10),
                         "recall@10": ev.recall_at_k(recs, test_hits[user], 10)})
    stop = pd.DataFrame(rows)
    stop.to_parquet(S_PATH, index=False)
    write_metadata(S_PATH, extended_list=[int(m) for m in ext_list], quota=EXT_QUOTA,
                   note="stop-on-likes onboarding; extended list starts with the official 30; "
                        "taste target = the user's pre-2018 likes not on the 100-movie list (same for every plan)")

agg = stop.groupby("plan").agg(median_questions=("questions", "median"), slowest_10pct=("questions", lambda s: s.quantile(0.9)),
                               likes=("likes", "mean"), taste_recall=("taste_recall@10", "mean"), future_recall=("recall@10", "mean"))
reach = stop[stop["plan"].str.startswith("stop at")].groupby("plan")["likes"].apply(
    lambda s: (s >= (20 if "20" in s.name else 30)).mean())
print(agg.round(4).to_string())
print("share of users reaching the target within 100 questions:", reach.round(3).to_dict())
idx = stop.set_index(["plan", "userId"])
same_name = [p for p in stop["plan"].unique() if p.startswith("fixed") and "official" not in p][0]
for a, b in [("stop at 20 likes", "fixed 30 (official)"), ("stop at 30 likes", "fixed 30 (official)"),
             ("stop at 20 likes", same_name), ("stop at 20 likes + dislikes", "stop at 20 likes")]:
    for metric in ("taste_recall@10", "recall@10"):
        d, lo, hi, rel = ev.paired_gain(idx.loc[a], idx.loc[b], metric, seed=SEED)
        print(f"{a} vs {b}: {metric} {rel:+.0%} [{lo:+.4f}, {hi:+.4f}]")

**Results (v3, fair grading; 5,000 simulated users)**

| Plan | Median questions | Slowest 10% | Taste recall@10 | vs official 30 |
|---|---|---|---|---|
| fixed 30 (official) | 30 | 30 | 0.0462 | — |
| fixed 54 (same median length) | 54 | 54 | 0.0522 | |
| **stop at 20 likes** | **54** | 100 (cap) | **0.0565** | **+22%** [+0.0087, +0.0119] |
| stop at 30 likes | 96 | 100 (cap) | 0.0580 | +26% [+0.0102, +0.0134] |
| stop at 20 likes + dislikes | 54 | 100 (cap) | 0.0489 | |

- **Stopping on likes is itself better than just asking more.** Against a fixed list of the same median length (54), stop-at-20-likes learns taste 8% better [+0.0031, +0.0055] and finds 11% more future favourites [+0.0002, +0.0024]. Users who like a lot stop early; users who like little keep going until there is enough to learn from.
- **20 likes is the better target.** Going to 30 likes adds only +4% more taste learning but needs a median 96 questions instead of 54, and only 52% of users reach 30 likes within 100 questions (69% reach 20). These are heavy raters; casual users would need longer.
- **Using dislikes the simple way hurts** (−13%): subtracting disliked vibes moves the taste in the wrong direction (a disliked movie usually shares much with liked ones, e.g. the same genre). Rejected in this form; a learned weight could be tried later.
- Future-hit gains against the official 30 are small and not significant (+3% to +9%), for the same reason as before: heavy test users have already seen the classics.

**Decision (approved by Adil, 2026-10-09):** stop at **20 likes**, with a cap of 100 questions and an "I'm done" option at any time. Answers: like / dislike / no clue. The question order is the extended list, whose first 30 are the official questionnaire.